# Lab 6 · Notebook 3: From one crop to every crop

**Time:** about 40 minutes. **What you will do:** answer the lecture's question for all 18 crops at once. If the wild bumblebees disappeared, which crop would lose the most?

This notebook walks through steps 2 to 5 of the lecture, then takes one step further: checking that the answer is fair.

Same format as before: **Demo**, **Write it** (with a self-check) and **Broken** cells.

## A reminder of the data

**One row is one survey**: one patch of one crop, watched for a fixed time, with every insect visit counted. Today you need `Crop` (`Rbean` = runner bean, `Bbean` = broad bean), `Allotment`, `Month`, `Honeybees` and `Bumblebees`.

In notebook 2 you did step 1 for one crop at a time: broad bean got 23 bumblebee visits and only 2 honeybee visits. Now for every crop.

In [ ]:
# Run this first. It loads the survey you used in the lecture.
import urllib.request
from pathlib import Path

import pandas as pd

FILE = "pollinators_lecture.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/" + FILE

# On Colab the file isn't there yet, so download it once.
if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE)
print("Loaded", FILE, df.shape)

## 1. groupby: one answer per group

`groupby` does three things in one line:

1. **Split** the table into piles, one pile per value in a column. Grouping by `Crop` makes 18 piles, one per crop.
2. **Take** the column you are interested in from each pile.
3. **Combine**: work out one number per pile (for example, add it up) and list the answers together.

```python
df.groupby("Crop")["Bumblebees"].sum()
#          split       take         combine
```

You can read it as a sentence: *"for each crop, take the bumblebee counts, and add them up."*

The column you group by is called the **key**. In the result, the key's values (the crop names) become the row labels on the left: the **index**.

In [ ]:
# DEMO - total bumblebee visits for each allotment
per_allotment = df.groupby("Allotment")["Bumblebees"].sum()
per_allotment

Nine allotments, nine piles, nine totals, listed in alphabetical order. Roedale had the most bumblebee visits (150). Camp_Site had none, but remember from notebook 1 that it was only surveyed twice.

You can pick one value out of the result by its label, the same way as with `value_counts()`: `per_allotment["Roedale"]`.

### Write it

**Predict first:** which month do you expect to have the most bumblebee visits? (Think about when there are most flowers, and when bumblebee nests are biggest.)

**Your task:** make `per_month`, the total bumblebee visits for each month. Then store the total for **August** in `august`.

In [ ]:
# TODO - bumblebee visits per month


# --- self-check (do not edit) ---
for name in ["per_month", "august"]:
    assert name in globals(), f"Make a variable called {name}"
assert len(per_month) == 6, f"per_month should have 6 rows (one per month), got {len(per_month)}. Group by 'Month'"
assert august == 190, f"august should be 190, got {august}. Try per_month['August']"
print("Correct: 190 bumblebee visits in August, the most of any month")

> The months come out in alphabetical order (April, August, July, …), not calendar order, because pandas doesn't know they are months. That doesn't change any of the numbers.

## 2. Several columns at once (lecture step 2)

To get the totals for **both** kinds of bee, take a list of columns instead of one, with double square brackets, as you did in notebook 2:

```python
visits = df.groupby("Crop")[["Honeybees", "Bumblebees"]].sum()
```

The result is a small DataFrame: one row per crop, one column per kind of bee. This is the table the rest of the notebook builds on.

In [ ]:
# DEMO - honeybee and bumblebee visits for every crop
visits = df.groupby("Crop")[["Honeybees", "Bumblebees"]].sum()
visits

18 rows, one per crop. Find `Bbean`: 2 and 23, the same numbers you worked out by hand in notebook 2.

### Write it

**Your task:** make `by_site`, the honeybee **and** bumblebee totals for each **allotment**. Then look at it: is there an allotment where honeybees outnumber bumblebees by more than two to one?

In [ ]:
# TODO - honeybee and bumblebee totals per allotment


# --- self-check (do not edit) ---
assert "by_site" in globals(), "Make a variable called by_site"
assert list(by_site.columns) == ["Honeybees", "Bumblebees"], f"by_site should have the columns Honeybees and Bumblebees, got {list(by_site.columns)}"
assert len(by_site) == 9, f"by_site should have one row per allotment (9), got {len(by_site)}. Group by 'Allotment'"
assert by_site.loc["Weald", "Honeybees"] == 259, "Weald should have 259 honeybee visits. Check that you used .sum()"
print("Correct. Weald: 259 honeybee visits against 129 bumblebee visits")

## 3. A new column: the bumblebee share (lecture step 3)

The totals don't answer the question yet. Raspberry has the most bumblebee visits, but it also has lots of honeybee visits, so it would cope without the bumblebees. What matters is **what share of each crop's bee visits come from bumblebees**:

$$\text{share} = \frac{\text{bumblebee visits}}{\text{honeybee visits} + \text{bumblebee visits}}$$

You make a new column by assigning to a column name that doesn't exist yet. The calculation is done **row by row**, for every crop at once, just like arithmetic on a numpy array:

```python
visits["share"] = visits["Bumblebees"] / (visits["Honeybees"] + visits["Bumblebees"])
```

The round brackets make the addition happen before the division, as in maths.

In [ ]:
# DEMO - the bumblebee share for every crop
visits["share"] = visits["Bumblebees"] / (visits["Honeybees"] + visits["Bumblebees"])
visits.round(3)

Broad bean: 23 / (2 + 23) = 0.92. So 92 out of every 100 bee visits to broad bean were bumblebees.

Look at **Pepper**: it shows `NaN`. No bees of either kind were seen on pepper, so the sum is 0 / (0 + 0), and you can't divide by zero. `NaN` stands for "Not a Number": pandas' way of saying "there is no answer here". It is not the same as 0.

### Write it

A share between 0 and 1 is easier to read as a percentage.

**Your task:** add a column `percent` to `visits` that holds the share as a percentage, **rounded to 1 decimal place**. Multiply by 100, then use `.round(1)` on the result.

In [ ]:
# TODO - a percent column


# --- self-check (do not edit) ---
assert "percent" in visits.columns, "visits has no column called percent yet. Assign it: visits['percent'] = ..."
assert visits.loc["Bbean", "percent"] == 92.0, f"Broad bean should be 92.0, got {visits.loc['Bbean', 'percent']}. Multiply the share by 100"
assert visits.loc["Raspberry", "percent"] == 55.9, f"Raspberry should be 55.9, got {visits.loc['Raspberry', 'percent']}. Did you round to 1 decimal place?"
print("Correct: broad bean 92.0%, raspberry 55.9%")

`visits.loc["Bbean", "percent"]`, used in the self-check above, picks out a single value from a table: the row label first, then the column name. You'll use it again below.

## 4. Sorting to find the top crop (lecture step 4)

To see which crop has the highest share, sort the table by that column, biggest first, as in notebook 2:

```python
visits.sort_values("share", ascending=False)
```

### Write it

**Predict first:** you know broad bean's share is 92%. Will broad bean be at the top?

**Your task:** make `ranked`, the `visits` table sorted by `share` with the highest first. Then store the name of the crop in the **first** row in `top_crop`. The crop names are in the index, so `ranked.index[0]` gives the first one.

In [ ]:
# TODO - sort by share and find the top crop


# --- self-check (do not edit) ---
for name in ["ranked", "top_crop"]:
    assert name in globals(), f"Make a variable called {name}"
assert ranked["share"].iloc[0] == 1.0, "ranked should start with the highest share. Did you use ascending=False?"
assert top_crop in ("Strawberry", "Blueberry"), f"top_crop should be the crop in the first row of ranked, got {top_crop!r}"
print(f"pandas says: {top_crop}, with a share of 100%. Look at its numbers before you believe it.")

Broad bean is not at the top. **Strawberry** and **blueberry** both have a share of 100%. Look at their rows in `ranked`: each had exactly **one** bee visit, and it happened to be a bumblebee. One visit out of one is 100%, but it tells you almost nothing about these crops.

This is the same problem as Camp_Site in notebook 1: a number based on very little data can be extreme just by chance.

## 5. Can we trust it? (lecture step 5)

The fix is to keep only the crops with enough data. As in the lecture, we keep crops with **at least 20 bumblebee visits**:

```python
enough = visits[visits["Bumblebees"] >= 20]
```

This is the same filtering as in notebook 2: a mask inside square brackets. It works on any DataFrame, including one you made with `groupby`.

Why 20? There is no magic number. What matters is that you choose the cut-off **before** you look at which crop wins, so you can't pick one that gives the answer you hoped for.

### Write it

**Your task:**

1. Make `enough`, the rows of `visits` with at least 20 bumblebee visits.
2. Sort it by `share`, highest first, and store the result in `trusted`.
3. Store the name of the crop in the first row of `trusted` in `answer`.

In [ ]:
# TODO - keep the crops with enough data, then sort


# --- self-check (do not edit) ---
for name in ["enough", "trusted", "answer"]:
    assert name in globals(), f"Make a variable called {name}"
assert len(enough) == 5, f"enough should have 5 crops, got {len(enough)}. Use visits['Bumblebees'] >= 20"
assert answer == "Bbean", f"answer should be 'Bbean', got {answer!r}. Did you sort trusted by share, highest first?"
print("Correct: once the tiny crops are out, broad bean is top, with 92% of its bee visits from bumblebees")

That is the lecture's answer, found by you: **broad bean**. If the bumblebees disappeared, broad bean would lose 92% of its bee visits. Raspberry would lose about 56%, and still have plenty of honeybees.

### Broken

This should keep the crops with at least 20 bumblebee visits, but the result is wrong. There is no error message this time: read the code carefully and compare it with step 5.

In [ ]:
# BROKEN - crops with at least 20 bumblebee visits
fixed = visits["Bumblebees"] >= 20
print(fixed)


# --- self-check (do not edit) ---
assert len(fixed.shape) == 2, "fixed is still a column of True/False values, not a table. Put the mask inside visits[ ... ]"
assert len(fixed) == 5
print("Fixed. A mask on its own is just True/False per row. It only filters when you put it inside the table's square brackets.")

## 6. Comparing like with like

There is one more thing to check, from the last slide of the lecture. Some crops were surveyed far more often than others: raspberry 269 times, blackberry only 57. Any **total** favours the crops that were watched most.

The share doesn't have this problem (it compares two counts from the same surveys). But if you want to ask *"which crop do bumblebees visit most?"*, you need visits **per survey**:

$$\text{bumblebees per survey} = \frac{\text{bumblebee visits}}{\text{number of surveys}}$$

`.size()` counts the rows in each group, so `df.groupby("Crop").size()` gives the number of surveys per crop. Because `visits` has the crops as its row labels too, you can put that straight into a new column and pandas lines the crops up for you.

In [ ]:
# DEMO - how many surveys of each crop?
visits["surveys"] = df.groupby("Crop").size()
visits[["Bumblebees", "surveys"]].head()

### Write it

**Your task:**

1. Add a column `bb_per_survey` to `visits`: bumblebee visits divided by the number of surveys.
2. Store the crop with the highest `bb_per_survey` in `busiest` (sort, then take `.index[0]`).

In [ ]:
# TODO - bumblebee visits per survey


# --- self-check (do not edit) ---
assert "bb_per_survey" in visits.columns, "visits has no column called bb_per_survey yet"
assert round(visits.loc["Bbean", "bb_per_survey"], 3) == 0.245, "Broad bean should have 23 / 94 = 0.245 bumblebees per survey. Divide Bumblebees by surveys"
assert "busiest" in globals(), "Make a variable called busiest"
assert busiest == "Blackberry", f"busiest should be 'Blackberry', got {busiest!r}. Sort by bb_per_survey, highest first"
print("Correct: blackberry gets the most bumblebee visits per survey (1.25), then raspberry (1.16). Broad bean gets 0.25.")

## Two questions, two answers

This is the most important result in the notebook, so it is worth reading slowly.

- **Which crop depends most on bumblebees?** Broad bean. Almost all its bee visits (92%) are bumblebees, so it would lose almost everything if they disappeared. This is the lecture's question, and the **share** answers it.
- **Which crop do bumblebees visit most often?** Blackberry and raspberry, at more than one bumblebee per survey. Broad bean gets only 0.25 per survey. The **visits per survey** answer this one.

Both answers are correct. They answer different questions, so before you calculate anything, decide which question you are asking.

## What you can do now

- `df.groupby("A")["B"].sum()`: one total of B for each value of A (split, take, combine)
- `df.groupby("A")[["B", "C"]].sum()`: the same for several columns, giving a table
- `df.groupby("A").size()`: how many rows are in each group
- `table["new"] = ...` to add a column calculated from other columns, row by row
- `.sort_values("col", ascending=False)` and `.index[0]` to find the top row
- a mask inside square brackets to keep only the groups with enough data

You have now done all five steps of the lecture yourself. **Lecture 7** turns tables like `visits` into figures.

## If you finish early

Optional, with no self-checks.

1. Per survey isn't the only fair comparison. Make a column of bumblebee visits **per 100 flowers**: you'll need `df.groupby("Crop")["Flowers"].sum()`. Which crop comes top now? Look at how many bumblebee visits and how many flowers that answer is based on. Do you trust it? (Remember from notebook 1 how the flowers were counted.)
2. In which months did the bumblebees visit broad bean? Filter to broad bean first, then group by month.
3. `.agg()` does several summaries in one go. Try `df.groupby("Crop")["Bumblebees"].agg(["sum", "mean", "size"])` and work out what each column means.

In [ ]:
# Your answers here